## 1. Data loading & Initial inspection

In [ ]:
import pandas as pd

import numpy as np

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

df.head(30)

In [ ]:
df.info()

In [ ]:
df.shape

## 2. Data profiling & Quality inspection: Inspect the structure, missing values, hidden whitespace, and distinct categorical values.


In [ ]:

data_quality_report = pd.DataFrame({
    'Data_Type': df.dtypes,
    'Null_Count': df.isnull().sum(),
    'Blank_Space_Count': (df == ' ').sum(),
    'Unique_Values': df.nunique()
})

data_quality_report

In [ ]:
# Inspect value counts for all categorical features
for col in df.select_dtypes(include=['object']).columns:
    print(f"=== Column: {col} (Unique: {df[col].nunique()}) ===")
    print(df[col].value_counts(dropna=False))
    print("-" * 50)

In [ ]:
# Summary statistics for numerical columns
df.describe().T

### Key profiling findings:
1. **SeniorCitizen**: Stored as numeric `0`/`1`; needs mapping to `'No'`/`'Yes'` for readability.
2. **TotalCharges**: Stored as `object` data type instead of numeric due to 11 blank spaces (`' '`)
3. **Categorical Features**: Service columns contain clean and consistent categories (`Yes`, `No`, `No internet service` / `No phone service`).

## 3. Data cleaning

In [ ]:
#SeniorCitizen

df['SeniorCitizen'] = df['SeniorCitizen'].map({1: 'Yes', 0: 'No'})


In [ ]:
#TotalCharges: 11 blank spaces 

blank_rows = df[df['TotalCharges'] == ' '][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]
blank_rows

In [ ]:
blank_rows.shape

In [ ]:
len(df[df['tenure'] == 0])


In [ ]:

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce')

df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])



In [ ]:
# Verify data type and check for remaining missing values
print("TotalCharges Data Type:", df['TotalCharges'].dtype)
print("Missing values in TotalCharges:", df['TotalCharges'].isnull().sum())

In [ ]:
df.info()

## 4. Feature engineering
In this section, I create new features from existing columns to support cohort analysis and visual slicing in Power BI.

In [ ]:
# Create tenure groups
labels = ['< 1 Year', '1-2 Years', '2-3 Years', '3-4 Years', '4-5 Years', '5-6 Years']
df['Tenure_Group'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 36, 48, 60, 72], labels=labels)

# Inspect the newly created column
df[['tenure', 'Tenure_Group']].head(10)

## 5. Dataset export

In [ ]:
df.to_csv('telco_customer_churn_cleaned.csv', index=False)

In [ ]:
df.info

## 6. Baseline KPI Metrics

In [ ]:
df['Churn_Numeric'] = (df['Churn'] == 'Yes').astype(int)

# Calculate baseline metrics
total_customers = len(df)
churned_customers = df['Churn_Numeric'].sum()
churn_rate = (churned_customers / total_customers) * 100
total_revenue = df['TotalCharges'].sum()
lost_revenue = df[df['Churn'] == 'Yes']['TotalCharges'].sum()

print("=== BASELINE KPIS ===")
print(f"Total Customers: {total_customers:,}")
print(f"Churned Customers: {churned_customers:,}")
print(f"Overall Churn Rate: {churn_rate:.2f}%")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Lost Revenue: ${lost_revenue:,.2f}")

## 7. Feature screening 

In [ ]:
all_categorical_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 
    'PhoneService', 'MultipleLines', 'InternetService', 
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
    'TechSupport', 'StreamingTV', 'StreamingMovies', 
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

feature_ranking = []

for col in all_categorical_cols:
    grouped = df.groupby(col)['Churn_Numeric'].mean() * 100
    max_rate = grouped.max()
    min_rate = grouped.min()
    variance = max_rate - min_rate
    
    feature_ranking.append({
        'Feature': col,
        'Min_Churn_%': round(min_rate, 2),
        'Max_Churn_%': round(max_rate, 2),
        'Variance_%': round(variance, 2)
    })

ranking_df = pd.DataFrame(feature_ranking).sort_values(by='Variance_%', ascending=False).reset_index(drop=True)

print("=== AFFECTION RANKING ===")
print(ranking_df)

High-impact areas (Variance > 30%): Contract type, payment method, internet service, and tech/security add-ons 

Dashboard filter candidates (Variance 13%–18%): SeniorCitizen, PaperlessBilling, Partners, and Dependents show a noticeable difference in churn

Safe to drop (Variance < 5%): Gender, phone service, and multiple lines 

## 8. High-Impact operational & Financial analysis

In [ ]:
# 1. Contract & Tenure group analysis
print("=== 1. CHURN RATE BY CONTRACT & TENURE GROUP ===")
contract_tenure = df.groupby(['Contract', 'Tenure_Group'])['Churn_Numeric'].agg(
    Total_Customers='count',
    Churned_Customers='sum',
    Churn_Rate_Pct=lambda x: round(x.mean() * 100, 2)
).reset_index()

print(contract_tenure.sort_values(by='Churn_Rate_Pct', ascending=False).to_string(index=False))



In [ ]:
# 2. Retention impact of Add-on services (Internet Users Only)
print("\n=== 2. RETENTION IMPACT OF ADD-ON SERVICES (INTERNET USERS ONLY) ===")
internet_users = df[df['InternetService'] != 'No']
addons = ['OnlineSecurity', 'TechSupport', 'OnlineBackup', 'DeviceProtection', 'StreamingTV', 'StreamingMovies']

addon_impact = []
for col in addons:
    rates = internet_users.groupby(col)['Churn_Numeric'].mean() * 100
    no_serv = round(rates.get('No', 0), 2)
    yes_serv = round(rates.get('Yes', 0), 2)
    addon_impact.append({
        'Addon_Service': col,
        'Churn_Pct_Without': no_serv,
        'Churn_Pct_With': yes_serv,
        'Risk_Reduction_Pct': round(no_serv - yes_serv, 2)
    })

addon_df = pd.DataFrame(addon_impact).sort_values(by='Risk_Reduction_Pct', ascending=False)
print(addon_df.to_string(index=False))



In [ ]:
# 3. Financial leakage 
print("\n=== 3. TOP FINANCIAL LEAKAGE HOTSPOTS ===")
leakage = df[df['Churn'] == 'Yes'].groupby(['InternetService', 'PaymentMethod']).agg(
    Lost_Customers=('customerID', 'count'),
    Total_Lost_Revenue=('TotalCharges', 'sum')
).reset_index()

total_lost = df[df['Churn'] == 'Yes']['TotalCharges'].sum()
leakage['Revenue_Loss_Pct'] = round((leakage['Total_Lost_Revenue'] / total_lost) * 100, 2)
print(leakage.sort_values(by='Total_Lost_Revenue', ascending=False).to_string(index=False))

# Key takeaways 

Month to month contracts in year one present the highest risk
Customers on month to month plans during their first year show the steepest churn rates. Churn drops significantly once a customer stays beyond one year or switches to an annual agreement.

Tech support and security act as core retention anchors
Internet subscribers without TechSupport or OnlineSecurity churn at more than double the rate of those who have them. Promoting these add ons during onboarding is a direct lever for retention.

Fiber optic with electronic check is the main financial leakage point
The combination of Fiber Optic internet and Electronic Check payments accounts for the single largest share of revenue lost. Operational priorities should focus on improving fiber service stability and encouraging automated payment methods.